# Практическое занятие № 18 — PCA, SVD и влияние снижения размерности на кластеризацию
**ТюмГУ · «Технологии искусственного интеллекта — 1» · 90 минут**

Цель: вручную проверить PCA на матрице 4×2, затем освоить масштабирование, объяснённую дисперсию, восстановление и сравнение кластеризации на Wine (13 признаков) и заказах. **Нельзя** обучать scaler или PCA на holdout и использовать целевые метки для подбора признаков/компонент.

## 1. Вывод PCA на маленьком примере (0–25 мин)
Для центрированной матрицы $X_c$ выборочная ковариация $\Sigma=X_c^T X_c/(n-1)$. Главная ось решает $\max_{\|w\|=1} w^T\Sigma w$; $Z=X_cW_k$. При SVD $X_c=U S V^T$ собственные значения ковариации равны $\lambda_j=s_j^2/(n-1)$.

В примере лекции собственные значения `6` и `2/3`; первая компонента объясняет 90% дисперсии. Знак собственных векторов произволен. Проверьте вычисления самостоятельно.

In [ ]:
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, mean_squared_error
print('Python:',sys.version.split()[0]); import sklearn; print('sklearn:',sklearn.__version__)

X_small=np.array([[2.,1.],[1.,2.],[-1.,-2.],[-2.,-1.]])
X_c=X_small-X_small.mean(axis=0)
Sigma=X_c.T @ X_c/(len(X_c)-1)
eigvals,eigvecs=np.linalg.eigh(Sigma)
order=np.argsort(eigvals)[::-1]; eigvals=eigvals[order]; eigvecs=eigvecs[:,order]
U,S,Vt=np.linalg.svd(X_c,full_matrices=False)
print('Среднее:',X_small.mean(axis=0));print('Ковариационная матрица:\n',Sigma)
print('Собственные значения:',eigvals,'Доли:',eigvals/eigvals.sum())
print('SVD singular values:',S,'S**2/(n-1):',S**2/(len(X_small)-1))
assert np.allclose(eigvals,[6,2/3]) and np.isclose(eigvals[0]/eigvals.sum(),.9)
assert np.allclose(eigvals,S**2/(len(X_small)-1))
pca_small=PCA(n_components=1).fit(X_small)
proj=pca_small.transform(X_small); recovered=pca_small.inverse_transform(proj)
rmse_small=np.sqrt(np.mean((X_small-recovered)**2))
print('RMSE восстановления по одной PC:',round(rmse_small,4))
fig,ax=plt.subplots(figsize=(6,4.6));ax.scatter(X_small[:,0],X_small[:,1],s=65,label='Исходные')
ax.scatter(recovered[:,0],recovered[:,1],marker='x',s=100,label='Восстановленные по PC1')
for i in range(len(X_small)):
    ax.plot([X_small[i,0],recovered[i,0]],[X_small[i,1],recovered[i,1]],'--',lw=1,color='gray')
ax.axhline(0,c='gray',lw=.5);ax.axvline(0,c='gray',lw=.5)
ax.set(xlabel='x1',ylabel='x2',title='Проекция на PC1 и обратное восстановление');ax.legend();ax.grid(alpha=.2);plt.show()

**Самостоятельно:** выпишите $w_1$ с точностью до знака; объясните, почему все данные, восстановленные по одной PC, лежат на прямой. Что теряется при удалении PC2?

## 2. Wine: 13 исходных измерений и масштабирование без утечек (25–42 мин)
Используйте встроенный набор `sklearn.datasets.load_wine` (178 объектов, 13 признаков). Истинные метки `wine.target` не участвуют в построении PCA и K-means; они могут использоваться **только для отдельной последующей проверки**, если преподаватель разрешит. Случайно отложите 20% объектов. Все параметры масштабирования — только по train.

In [ ]:
wine=load_wine(as_frame=True)
X=wine.data.copy();true_labels=wine.target.to_numpy()
idx_train,idx_test=train_test_split(np.arange(len(X)),test_size=.20,random_state=42,shuffle=True)
X_train=X.iloc[idx_train].copy();X_test=X.iloc[idx_test].copy()
sc=StandardScaler().fit(X_train)
Z_train=sc.transform(X_train);Z_test=sc.transform(X_test)
assert len(X.columns)==13 and np.isfinite(Z_train).all() and np.isfinite(Z_test).all()
print('Wine:',X.shape,'train/test:',Z_train.shape,Z_test.shape)
print('Пример: средние по train после стандартизации:',Z_train.mean(axis=0).round(8)[:5])

pca_all=PCA().fit(Z_train)
ratio=pca_all.explained_variance_ratio_;cum=np.cumsum(ratio)
fig,axs=plt.subplots(1,2,figsize=(11,4),constrained_layout=True)
axs[0].bar(np.arange(1,len(ratio)+1),ratio);axs[0].set(xlabel='Номер PC',ylabel='Доля дисперсии',title='Explained variance ratio')
axs[1].plot(np.arange(1,len(cum)+1),cum,marker='o');axs[1].axhline(.95,ls='--',c='gray',label='95%')
axs[1].set(xlabel='Число компонент',ylabel='Накопленная доля',title='Cumulative explained variance');axs[1].legend()
for a in axs:a.grid(alpha=.2)
plt.show()
print('Первые 2 PC:',round(cum[1]*100,2),'% (значение для выбранного train может отличаться от всей выборки)')

## 3. PCA(2) и PCA(95%): удержанная дисперсия и ошибка восстановления (42–59 мин)
Порог `0.95` выбирает минимальное число компонент, объясняющее не менее 95% **обучающей** дисперсии. Реконструкцию измеряйте на `test` в **стандартизованных** координатах, чтобы признаки не перетягивали ошибку единицами измерения.

In [ ]:
pca2=PCA(n_components=2).fit(Z_train)
pca95=PCA(n_components=.95).fit(Z_train)
T2=pca2.transform(Z_train);T95=pca95.transform(Z_train)
T2_test=pca2.transform(Z_test);T95_test=pca95.transform(Z_test)
rec2=pca2.inverse_transform(T2_test);rec95=pca95.inverse_transform(T95_test)
rmse2=np.sqrt(np.mean((Z_test-rec2)**2));rmse95=np.sqrt(np.mean((Z_test-rec95)**2))
print('PCA 2:',T2.shape[1],'комп.; доля =',round(sum(pca2.explained_variance_ratio_),4),'test RMSE =',round(rmse2,4))
print('PCA 95%:',T95.shape[1],'комп.; доля =',round(sum(pca95.explained_variance_ratio_),4),'test RMSE =',round(rmse95,4))
assert T95.shape[1]>=2 and rmse95<=rmse2+1e-10
summary=pd.DataFrame([{'Метод':'PCA 2','Число PC':T2.shape[1],'Объяснённая доля':sum(pca2.explained_variance_ratio_),'test RMSE':rmse2},
                     {'Метод':'PCA 95%','Число PC':T95.shape[1],'Объяснённая доля':sum(pca95.explained_variance_ratio_),'test RMSE':rmse95}])
print(summary.to_string(index=False))

# Проверка численного соотношения для SVD на центрированном стандартизованном train
s=np.linalg.svd(Z_train-Z_train.mean(axis=0),full_matrices=False,compute_uv=False)
assert np.allclose(s**2/(len(Z_train)-1),pca_all.explained_variance_)
print('SVD ↔ собственные значения PCA: проверено.')

## 4. Как PCA влияет на кластеризацию? (59–78 мин)
Одни и те же объекты делим K-means на 3 группы до и после PCA. **ARI** сравнивает полученные разбиения и не зависит от номеров кластеров. Значения Silhouette в 13D и 2D отражают **разные геометрические пространства**, поэтому они не являются прямым доказательством «качество улучшилось».

In [ ]:
K=3
km_full=KMeans(n_clusters=K,n_init=10,random_state=42).fit(Z_train)
km2=KMeans(n_clusters=K,n_init=10,random_state=42).fit(T2)
km95=KMeans(n_clusters=K,n_init=10,random_state=42).fit(T95)
comp=pd.DataFrame([
 {'Пространство':'13D scaled','Silhouette':silhouette_score(Z_train,km_full.labels_),'ARI к 13D':1.0},
 {'Пространство':'PCA 2D','Silhouette':silhouette_score(T2,km2.labels_),'ARI к 13D':adjusted_rand_score(km_full.labels_,km2.labels_)},
 {'Пространство':'PCA 95%','Silhouette':silhouette_score(T95,km95.labels_),'ARI к 13D':adjusted_rand_score(km_full.labels_,km95.labels_)}
]);print(comp.to_string(index=False))
fig,ax=plt.subplots(figsize=(7,5))
ax.scatter(T2[:,0],T2[:,1],c=km2.labels_,s=33,cmap='tab10',alpha=.85)
ax.scatter(km2.cluster_centers_[:,0],km2.cluster_centers_[:,1],marker='X',s=200,c='black',label='Центроиды')
ax.set(xlabel=f'PC1 ({pca2.explained_variance_ratio_[0]:.1%})',ylabel=f'PC2 ({pca2.explained_variance_ratio_[1]:.1%})',title='Wine: кластеры K-means в координатах PCA(2)')
ax.legend();ax.grid(alpha=.2);plt.show()

# Внепроцедурная оценка по известным меткам (не влияет ни на обучение, ни на подбор):
print('Внешний ARI с известными категориями Wine (только post-hoc):',round(adjusted_rand_score(true_labels[idx_train],km2.labels_),4))

**Самостоятельно:** повторите K-means при `K=2` и `K=4`, назовите, какие сравнения правомерны. Объясните, почему ортогональный поворот без уменьшения числа компонент сохраняет евклидовы расстояния, а обрезка до 2PC — нет.

## 5. Возвращение к сквозному кейсу заказов (78–90 мин)
По трём доступным при создании заказа признакам оцените PCA(2) и различие разбиений K-means **только на ранних 80% заказов**. Поля `actual_minutes`, `late` и `order_id` исключены из пространства признаков. Не переносите обученный PCA с Wine на заказы — распределения и смысл признаков не совпадают.

In [ ]:
source=Path('orders_dirty.csv')
if not source.exists():source=Path('demo_orders_17_18.csv')
if not source.exists():
    print('Файл заказов не загружен; выполните этот пункт после добавления orders_dirty.csv / demo_orders_17_18.csv')
else:
    orders=pd.read_csv(source,encoding='utf-8-sig').drop_duplicates().copy()
    FEATURES=['distance_km','queue_size','promised_minutes']
    assert {'order_id','created_at',*FEATURES}.issubset(orders.columns)
    orders['created_at']=pd.to_datetime(orders.created_at,errors='coerce')
    orders=orders.dropna(subset=['created_at','order_id']).sort_values('created_at').drop_duplicates('order_id').reset_index(drop=True)
    for c in FEATURES:orders[c]=pd.to_numeric(orders[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
    train_orders=orders.iloc[:int(len(orders)*.8)].copy();holdout_orders=orders.iloc[int(len(orders)*.8):].copy()
    from sklearn.impute import SimpleImputer
    imp=SimpleImputer(strategy='median').fit(train_orders[FEATURES])
    order_sc=StandardScaler().fit(imp.transform(train_orders[FEATURES]))
    Z_order=order_sc.transform(imp.transform(train_orders[FEATURES]))
    pca_order=PCA(n_components=2).fit(Z_order);T_order=pca_order.transform(Z_order)
    k_o_full=KMeans(n_clusters=3,n_init=10,random_state=42).fit_predict(Z_order)
    k_o_2=KMeans(n_clusters=3,n_init=10,random_state=42).fit_predict(T_order)
    print('Источник:',source.name, '| train:',len(train_orders),'| holdout:',len(holdout_orders))
    print('Объяснённая доля 2 PC для заказов:',round(sum(pca_order.explained_variance_ratio_),4))
    print('ARI разбиений заказов 3D и PCA 2D:',round(adjusted_rand_score(k_o_full,k_o_2),4))
    result=train_orders[['order_id','created_at']].copy()
    result['cluster_3d']=k_o_full;result['cluster_pca2']=k_o_2
    result.to_csv('18_orders_cluster_comparison.csv',index=False)
summary.to_csv('18_wine_pca_summary.csv',index=False);comp.to_csv('18_wine_cluster_comparison.csv',index=False)
print('Сохранены CSV по Wine; по заказам — если файл был загружен.')

### Что сдавать
1. Отчёт 2–4 страницы: матрица ковариации и её спектр для 4×2, проверка SVD, таблица Wine «2PC/95%», график долей и 2D-разброса, таблица ARI сравнения кластеризаций.
2. `18_student.ipynb` (Restart & Run All), CSV при наличии; описание связи с кейсом заказов.
3. Самостоятельно: объясните роль стандартизации, знаковую неоднозначность компонент и почему 95% дисперсии **не** означает 95% качества ML-модели.

**Контроль:** отличается ли PCA от отбора двух исходных столбцов? Почему `pca.fit` нельзя делать на test? Какие потери при восстановлении по 2PC?